# Clinica t1-linear on IXI: installation and timing test

Goal: install Clinica on Colab, run `t1-linear` on **3 subjects (one per hospital)**, and measure the time per subject before processing all 563 images.

t1-linear = N4 bias field correction + affine registration to the MNI152NLin2009cSym template + crop (169 × 208 × 179, 1 mm).

**Runtime:** no GPU needed. Runtime → Change runtime type → **CPU**.

Clinica is installed in a **separate Python environment** (`/content/clinica_env`), so it cannot conflict with ClinicaDL.

## 1. Drive and scripts

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/test-clinicadl'
WORK = '/content/test-clinicadl'
REPO = 'https://github.com/Ayoub-TAREK/ixi-brain-age.git'
if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !git -C {WORK} pull
os.makedirs(f'{WORK}/data/IXI/raw', exist_ok=True)
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(f'{DRIVE}/outputs', f'{WORK}/outputs')
!git -C {WORK} log --oneline -1
!nproc

## 2. Download IXI and convert to BIDS (same as the main notebook)

In [ ]:
import glob
!pip install -q xlrd
%cd {WORK}/data/IXI
URL = 'http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI'
if not os.path.exists('IXI.xls'):
    !curl -fL --retry 3 -o IXI.xls {URL}/IXI.xls
if len(glob.glob('raw/*.nii.gz')) < 500:
    !curl -fL --retry 3 -o IXI-T1.tar {URL}/IXI-T1.tar
    !tar -xf IXI-T1.tar -C raw && rm IXI-T1.tar
%cd {WORK}
!python scripts/ixi_to_bids.py

## 3. Install Clinica in a separate environment

`uv` creates an isolated Python 3.12 environment and installs Clinica 0.12.0 into it (a few minutes).

In [ ]:
CLINICA_ENV = '/content/clinica_env'
CLINICA = f'{CLINICA_ENV}/bin/clinica'
if not os.path.exists(CLINICA):
    !pip install -q uv
    !uv venv {CLINICA_ENV} --python 3.12 -q
    !uv pip install --python {CLINICA_ENV}/bin/python -q clinica==0.12.0
!{CLINICA} --version
!{CLINICA} run t1-linear --help

## 4. Timing test on 3 subjects (one per hospital)

Outputs go to `/content/caps_test` (temporary). `--use-antspy` avoids installing the ANTs binaries; `--random_seed 42` makes the registration reproducible.

In [ ]:
import time
import pandas as pd

BIDS = f'{WORK}/data/IXI/bids'
CAPS_TEST = '/content/caps_test'
table = pd.read_csv(f'{BIDS}/participants_sessions_age.tsv', sep='\t')
test = pd.concat([table[table.site == s].head(1) for s in ['Guys', 'HH', 'IOP']])
test[['participant_id', 'session_id']].to_csv('/content/test_subjects.tsv', sep='\t', index=False)
print(test[['participant_id', 'site', 'AgeMRI']].to_string(index=False))

N_PROCS = os.cpu_count()
start = time.time()
!{CLINICA} run t1-linear {BIDS} {CAPS_TEST} -tsv /content/test_subjects.tsv --use-antspy --random_seed 42 -np {N_PROCS} -wd /content/wd_test
elapsed = time.time() - start
print(f'\n3 subjects with {N_PROCS} processes: {elapsed / 60:.1f} min')
print(f'Rough estimate for 563 subjects: {elapsed / 3 * 563 / 3600:.1f} h (to be refined)')

## 5. Check the outputs

File names, sizes, image shape (expected 169 × 208 × 179) and three slices per subject. All three brains should be aligned and centered in the same way.

In [ ]:
%%writefile /content/show_t1_linear.py
import glob, sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import nibabel as nib

files = sorted(glob.glob(f'{sys.argv[1]}/subjects/*/*/t1_linear/*_desc-Crop_*_T1w.nii.gz'))
fig, axes = plt.subplots(len(files), 3, figsize=(10, 3.3 * len(files)), squeeze=False)
for row, f in zip(axes, files):
    img = nib.load(f)
    data = img.get_fdata()
    print(f.split('/')[-1], data.shape, img.header.get_zooms(), img.get_data_dtype())
    x, y, z = (s // 2 for s in data.shape)
    for ax, sl in zip(row, [data[x], data[:, y], data[:, :, z]]):
        ax.imshow(sl.T, cmap='gray', origin='lower')
        ax.axis('off')
    row[0].set_title(f.split('/')[-1].split('_ses')[0], loc='left')
fig.tight_layout()
fig.savefig(sys.argv[2], dpi=90)

In [ ]:
from IPython.display import Image, display
!find {CAPS_TEST}/subjects -type f | sort
!du -sh {CAPS_TEST}/subjects/*
!{CLINICA_ENV}/bin/python /content/show_t1_linear.py {CAPS_TEST} /content/t1_linear_test.png
display(Image('/content/t1_linear_test.png'))